# 01: Data Acquisition

## Goal
Download weekly ICU bed data for California from the CDC's National Healthcare Safety Network (NHSN) and save an unedited copy to `data/raw/`. Every later notebook builds from this file.

## Why this dataset
Hospitals across the US are required to report bed capacity and occupancy to the CDC every week. That makes this one of the few public sources that tracks ICU demand consistently over several years. It also comes with real-world problems: periods when reporting was optional, changes in how many hospitals reported, and the COVID-19 surge distorting the early years. Handling those issues is part of the work.

## Scope
- **Geography:** California, state level
- **Time unit:** one row per week
- **Forecasting target (later notebooks):** number of occupied ICU beds

## What I want to learn in this notebook
- How far back the data goes
- How many variables are available
- Whether anything in the raw file needs attention before cleaning

In [7]:
import sys
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Python: 3.14.8
pandas: 3.0.6


In [8]:
raw_data_folder = Path("../data/raw")
raw_data_folder.mkdir(parents=True, exist_ok=True)

nhsn_dataset_url = (
    "https://data.cdc.gov/resource/ua7e-t2fy.csv"
    "?jurisdiction=CA"
    "&$limit=5000"
    "&$order=weekendingdate"
)

california_weekly_raw = pd.read_csv(nhsn_dataset_url)

download_date = date.today().isoformat()
raw_file_path = raw_data_folder / f"nhsn_hrd_california_weekly_{download_date}.csv"
california_weekly_raw.to_csv(raw_file_path, index=False)

print("Saved to:", raw_file_path)
print("Rows, columns:", california_weekly_raw.shape)
print("First week:", california_weekly_raw["weekendingdate"].min())
print("Last week:", california_weekly_raw["weekendingdate"].max())

Saved to: ../data/raw/nhsn_hrd_california_weekly_2026-10-06.csv
Rows, columns: (321, 322)
First week: 2020-08-08T00:00:00.000
Last week: 2026-09-26T00:00:00.000


In [9]:
week_ending_dates = pd.to_datetime(california_weekly_raw["weekendingdate"])

duplicate_week_count = week_ending_dates.duplicated().sum()
days_between_rows = week_ending_dates.diff().dropna().dt.days

print("Duplicate weeks:", duplicate_week_count)
print("Gaps between consecutive rows (days):")
print(days_between_rows.value_counts())

Duplicate weeks: 0
Gaps between consecutive rows (days):
weekendingdate
7    320
Name: count, dtype: int64


In [10]:
column_names = california_weekly_raw.columns.tolist()
print(len(column_names), "columns\n")
for column_name in column_names:
    print(column_name)

322 columns

weekendingdate
jurisdiction
numinptbeds
numinptbedsadult
numinptbedsped
numinptbedsocc
numinptbedsoccadult
numinptbedsoccped
numicubeds
numicubedsadult
numicubedsped
numicubedsocc
numicubedsoccadult
numicubedsoccped
numconfc19hosppatsadult
numconfc19hosppatsped
totalconfc19hosppats
numconffluhosppatsadult
numconffluhosppatsped
totalconffluhosppats
numconfrsvhosppatsadult
numconfrsvhosppatsped
totalconfrsvhosppats
numconfc19icupatsadult
numconfc19icupatsped
totalconfc19icupats
numconffluicupatsadult
numconffluicupatsped
totalconffluicupats
numconfrsvicupatsadult
numconfrsvicupatsped
totalconfrsvicupats
numconfc19newadmped0to4
numconfc19newadmped5to17
totalconfc19newadmped
numconfc19newadmadult18to49
numconfc19newadmadult50to64
numconfc19newadmadult65to74
numconfc19newadmadult75plus
totalconfc19newadmadult
numconfc19newadmunk
totalconfc19newadm
numconfflunewadmped0to4
numconfflunewadmped5to17
totalconfflunewadmped
numconfflunewadmadult18to49
numconfflunewadmadult50to64
numco

## Result
- 321 rows and 322 columns, one row per week, from August 8, 2020 to September 26, 2026.
- No week appears twice, and every row is exactly 7 days after the one before it.

## Interpretation
- The data has every week in the date range, with nothing skipped or repeated. This matters because a forecast learns from week-to-week changes. A missing week would make two weeks of change look like one.
- Each date marks the last day of the week the row covers. Dates are stored as text with a time of midnight attached, which adds nothing and will be removed.
- The 322 columns fall into three groups:
  - **Raw counts** reported by hospitals, such as ICU beds available, ICU beds in use, and patients with COVID-19, flu, or RSV.
  - **Calculated values** built from those counts, such as percentages, rates per 100,000 people, and weekly changes.
  - **Reporting coverage**, meaning how many hospitals sent in each number that week.
- The forecast needs the ICU counts and the reporting coverage. The calculated values can be rebuilt from the raw counts if needed.

## Limitations
Having a row for every week does not mean every hospital reported that week. Totals depend on how many hospitals sent in data. If fewer hospitals report, ICU numbers drop even when the same number of patients need care. This is a known risk from May through October 2024, when reporting was optional. The reporting coverage columns will be used to separate real changes in ICU demand from changes in who reported.